# 원하는 형태로 정보 반환하기 (Structured Output)

LLM 의 답을 자유 텍스트가 아니라 **정해진 스키마(Pydantic 모델)** 로 받는 방법을 다룬다.
후반부에서는 도구로 정보를 가져온 뒤(arXiv 논문 검색), 그 결과를 구조화된 형태로 반환하는 에이전트를 만든다.

다루는 것:
1. `with_structured_output` 기본
2. 여러 스키마 중 적절한 것 선택 (Union)
3. 도구 호출 + 구조화 출력을 결합한 그래프
4. arXiv 도구로 논문 정보를 구조화해 반환

## 환경 변수 준비

이 노트북부터는 실제 LLM 을 호출하므로 API 키가 필요하다.
프로젝트 루트의 `.env` 파일에 키를 넣어두고 `load_dotenv()` 로 불러온다.

```
# .env  (.env.example 을 복사해서 작성)
OPENAI_API_KEY=sk-...
TAVILY_API_KEY=tvly-...   # 웹검색 노트북에서 필요
```

- OpenAI 키: <https://platform.openai.com/api-keys>
- Tavily 키: <https://app.tavily.com>

In [ ]:
import os
from dotenv import load_dotenv

# 프로젝트 루트의 .env 를 찾아 환경변수로 로드
load_dotenv()

# 필요한 키가 있는지 확인
assert os.environ.get("OPENAI_API_KEY"), "OPENAI_API_KEY 가 .env 에 없습니다"
print("환경변수 로드 완료:", "OPENAI_API_KEY")

## 1. `with_structured_output` 기본

Pydantic 모델을 정의하고 `model.with_structured_output(스키마)` 로 감싸면, LLM 이 그 스키마에 맞는 객체를 반환한다.

In [ ]:
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI

class MovieResponse(BaseModel):
    """영화 정보 응답 스키마"""
    title: str = Field(description="영화 제목")
    director: str = Field(description="감독 이름")
    genre: str = Field(description="장르")
    release_year: int = Field(description="개봉 연도")

model = ChatOpenAI(model="gpt-4o")
model_with_structured_output = model.with_structured_output(MovieResponse)

model_with_structured_output.invoke("타이타닉 영화에 대해 설명해주세요.")

## 2. 여러 스키마 중 선택하기 (Union)

영화 질문이면 `MovieResponse`, 잡담이면 `ConversationalResponse` 처럼 상황에 맞는 스키마를 LLM 이 고르게 할 수 있다.

In [ ]:
from typing import Union

class ConversationalResponse(BaseModel):
    """친근한 대화체 응답"""
    response: str = Field(description="사용자 질문에 대한 대화체 답변")

class FinalResponse(BaseModel):
    final_output: Union[MovieResponse, ConversationalResponse]

structured_llm = model.with_structured_output(FinalResponse)

In [ ]:
print(structured_llm.invoke("타이타닉에 대해 알려주세요."))
print(structured_llm.invoke("오늘 기분 어때?"))

## 3. 도구 호출 + 구조화 출력 그래프

흐름:
```
START → agent ──(도구 호출 있음)──> tools → agent
          │
          └──(도구 호출 없음)──> respond → END
```
agent 가 도구로 정보를 모으고, 더 부를 도구가 없으면 respond 노드가 결과를 구조화해 반환한다.

### Step 1. 출력 스키마

In [ ]:
class MovieResponse(BaseModel):
    """영화 정보 응답 스키마"""
    title: str = Field(description="영화 제목")
    director: str = Field(description="감독 이름")
    genre: str = Field(description="장르")
    release_year: int = Field(description="개봉 연도")

### Step 2. State 설정
`MessagesState` 는 `messages` 키를 기본 제공하는 State. 여기에 최종 구조화 결과 필드를 더한다.

In [ ]:
from langgraph.graph import MessagesState

class State(MessagesState):
    final_response: MovieResponse

### Step 3. 영화 정보 도구와 agent 노드

In [ ]:
from typing import Literal
from langchain_core.tools import tool

@tool
def get_movieinfo(movie: Literal["타이타닉", "노팅힐"]):
    """Use this to get movie information."""
    if movie == "타이타닉":
        return "영화 타이타닉은 1997년 개봉하였고, 감독은 제임스 카메론입니다. 장르는 로맨스/드라마입니다."
    elif movie == "노팅힐":
        return "영화 노팅힐은 1999년 개봉하였고, 감독은 로저 미첼입니다. 장르는 로맨틱 코미디입니다."
    else:
        raise AssertionError("Unknown movie")

tools = [get_movieinfo]
model_with_tool = model.bind_tools(tools)

def call_model(state: State):
    response = model_with_tool.invoke(state["messages"])
    return {"messages": [response]}

### Step 4. respond 노드와 분기
respond 는 도구 결과(직전 사용자/도구 메시지)를 구조화 스키마로 변환해 `final_response` 에 담는다.

In [ ]:
from langchain_core.messages import HumanMessage

model_with_structured_output = model.with_structured_output(MovieResponse)

def respond(state: State):
    # [-2]: ToolMessage(도구 결과), [-1]: AIMessage — 도구 결과를 구조화
    response = model_with_structured_output.invoke(
        [HumanMessage(content=state["messages"][-2].content)]
    )
    return {"final_response": response}

def should_continue(state: State):
    last_message = state["messages"][-1]
    if not last_message.tool_calls:
        return "respond"      # 더 부를 도구 없음 → 구조화 응답으로
    else:
        return "continue"     # 도구 호출 있음 → tools 실행


### Step 5. 그래프 생성

In [ ]:
from langgraph.graph import StateGraph, END
from langgraph.prebuilt import ToolNode

graph_builder = StateGraph(State)
graph_builder.add_node("agent", call_model)
graph_builder.add_node("respond", respond)
graph_builder.add_node("tools", ToolNode(tools))

graph_builder.set_entry_point("agent")
graph_builder.add_conditional_edges(
    "agent",
    should_continue,
    {"continue": "tools", "respond": "respond"},
)
graph_builder.add_edge("tools", "agent")
graph_builder.add_edge("respond", END)
graph = graph_builder.compile()

In [ ]:
from IPython.display import Image, display

try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception:
    print(graph.get_graph().draw_mermaid())

In [ ]:
answer = graph.invoke(
    input={"messages": [("human", "타이타닉 영화에 대해 알려주세요.")]}
)["final_response"]
print(answer)

## 4. arXiv 논문 정보를 구조화해 반환하기

이번엔 실제 외부 도구(arXiv API)로 논문을 가져와, 정해진 스키마로 반환한다.

먼저 arXiv 도구를 단독으로 써본다.

In [ ]:
from langchain_community.utilities import ArxivAPIWrapper

arxiv = ArxivAPIWrapper()
docs = arxiv.run("1605.08386")
print(docs)

### Step 1. 논문 출력 스키마

In [ ]:
class PaperResponse(BaseModel):
    """논문 정보 응답 스키마"""
    published: str = Field(description="논문 출판일")
    title: str = Field(description="논문 제목")
    authors: list[str] = Field(description="저자 목록")
    summary: str = Field(description="논문 요약")

class PaperState(MessagesState):
    final_response: PaperResponse

### Step 2. arXiv 도구와 노드

In [ ]:
@tool
def get_paper(arxivId: str):
    """Use this to get paper information."""
    return arxiv.run(arxivId)

tools = [get_paper]
model_with_tool = model.bind_tools(tools)

def call_model(state: PaperState):
    response = model_with_tool.invoke(state["messages"])
    return {"messages": [response]}

model_with_structured_output = model.with_structured_output(PaperResponse)

def respond(state: PaperState):
    response = model_with_structured_output.invoke(
        [HumanMessage(content=state["messages"][-2].content)]
    )
    return {"final_response": response}

def should_continue(state: PaperState):
    last_message = state["messages"][-1]
    return "respond" if not last_message.tool_calls else "continue"

### Step 3. 그래프 생성 및 실행

In [ ]:
graph_builder = StateGraph(PaperState)
graph_builder.add_node("agent", call_model)
graph_builder.add_node("respond", respond)
graph_builder.add_node("tools", ToolNode(tools))

graph_builder.set_entry_point("agent")
graph_builder.add_conditional_edges(
    "agent",
    should_continue,
    {"continue": "tools", "respond": "respond"},
)
graph_builder.add_edge("tools", "agent")
graph_builder.add_edge("respond", END)
graph = graph_builder.compile()

In [ ]:
answer = graph.invoke(
    input={"messages": [("human", "2312.15166 논문에 대해 알려줘.")]}
)["final_response"]

print("제목:", answer.title)
print("출판:", answer.published)
print("저자:", answer.authors)
print("요약:", answer.summary)

## 정리

- `with_structured_output(스키마)` = LLM 응답을 Pydantic 객체로 받기
- `Union` 스키마로 상황별 응답 형태 선택 가능
- **agent → tools → respond** 패턴: 도구로 정보 수집 후 마지막에 구조화 응답
- arXiv 같은 실제 외부 도구도 동일 패턴으로 결합

다음: 분기·번역·문법교정을 결합한 영어 회화 챗봇.